## Install and import required libraries

In [590]:
# !pip install scikit-learn
# !pip install numpy
# !pip install pandas
# !pip install matplotlib
# !pip install seaborn
# !pip install xgboost
# !pip install pyarrow

In [591]:
import pandas as pd
import numpy as np
import glob
import sqlite3
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [592]:
#load single file to look at contents
df = pd.read_csv('data/ids_0.csv')

#get names of columns
print(df.columns)

#display data types
print(df.dtypes)

#disply dataframe head
df.head()

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,53,61205,4,2,136,428,34,34,34.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,222,2,2,90,172,45,45,45.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,53,23759,2,2,70,126,35,35,35.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,80,401,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,57406,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [593]:
#combine all csv, json, and parquet files into single dataframe

def extract() -> pd.DataFrame:
    #main dataframe that everything will be concatenated to
    data = pd.DataFrame()

    #extract CSV files
    for csvfile in glob.glob('data/*.csv'):
        tmp_df = pd.read_csv(csvfile)
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    #extract JSON files
    for jsonfile in glob.glob('data/*.json'):
        tmp_df = pd.read_json(jsonfile, lines=True)
        data = pd.concat([data, tmp_df], ignore_index=True)

    #extract Parquet files
    for parquetfile in glob.glob('data/*.parquet'):
        tmp_df = pd.read_parquet(parquetfile)
        data = pd.concat([data, tmp_df], ignore_index=True)
    
    #return combined dataframe
    return data

In [594]:
#call extract function to combine all files into single dataframe
df = extract()

print("Shape of combined dataframe:", df.shape)

df.head()

Shape of combined dataframe: (63129, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,55109,17,1,1,6,6,6,6,6.0,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,53,113594958,4,4,152,362,45,31,38.0,8.082904,...,32,240.0,0.0,240,240,114000000.0,0.0,114000000,114000000,BENIGN
2,53,30485,1,1,81,209,81,81,81.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,53,30445,1,1,53,81,53,53,53.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,53,70860,1,1,56,72,56,56,56.0,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [595]:
#verify the extracted data
#remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()  

#check label distribution
print(df['Label'].value_counts())

Label
DoS Hulk            31027
DoS GoldenEye       20586
BENIGN               6006
DoS Slowhttptest     5499
Heartbleed             11
Name: count, dtype: int64


In [596]:
#Tranform 

def transform(df: pd.DataFrame) -> pd.DataFrame:

    #remove Heartbleed rows since its not a DOS attack
    df = df[df['Label'] != 'Heartbleed']

    #remove any duplicate rows
    df = df.drop_duplicates()

    #remap labels (benign stays but all dos attacks get remapped to "attack")
    df['Label'] = df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'attack')

    #replace infinite values with NaN and then drop rows with NaN values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    return df


In [597]:
#call transform function to clean the data
df = transform(df)
print("Shape of transformed dataframe:", df.shape)

#check label distribution after transformation
print(df['Label'].value_counts())

Shape of transformed dataframe: (49348, 79)
Label
attack    44402
BENIGN     4946
Name: count, dtype: int64


In [598]:
#load the data into a csv file
df.to_csv('data/cleaned_data.csv', index=False)

In [599]:
#read in cleaned csv file as a dataframe
df = pd.read_csv('data/cleaned_data.csv')
print(df.shape)

(49348, 79)


In [600]:
#exploratory data analysis - are there NA values? 
print(df.isna().sum())
#all 0's = no NA values

Destination Port               0
Flow Duration                  0
Total Fwd Packets              0
Total Backward Packets         0
Total Length of Fwd Packets    0
                              ..
Idle Mean                      0
Idle Std                       0
Idle Max                       0
Idle Min                       0
Label                          0
Length: 79, dtype: int64


In [601]:
#find correlation values between features
df.iloc[:, :-1].corr().head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
Destination Port,1.000000,-0.147549,-0.005291,-0.003627,0.028126,-0.003533,0.037428,0.002504,0.080888,0.039869,...,-0.001634,-0.072640,-0.041362,-0.011385,-0.041389,-0.039745,-0.126494,-0.027524,-0.130626,-0.121881
Flow Duration,-0.147549,1.000000,0.019596,0.018132,0.050770,0.016627,0.144603,-0.113551,0.026332,0.163949,...,0.015314,-0.247093,0.076993,0.084449,0.091520,0.060206,0.939545,0.178101,0.963662,0.909712
Total Fwd Packets,-0.005291,0.019596,1.000000,0.999252,0.506511,0.998954,0.021466,-0.004608,-0.000587,0.003699,...,0.998759,-0.012167,0.005936,0.010295,0.008545,0.003995,-0.004477,0.001721,-0.004184,-0.004642
Total Backward Packets,-0.003627,0.018132,0.999252,1.000000,0.506405,0.998760,0.021128,-0.003027,-0.000559,0.004233,...,0.998627,-0.019500,-0.005199,0.003021,-0.003373,-0.005703,-0.002386,-0.004159,-0.003062,-0.001886
Total Length of Fwd Packets,0.028126,0.050770,0.506511,0.506405,1.000000,0.494377,0.371997,0.020882,0.351646,0.336104,...,0.504996,-0.024983,0.033697,0.230855,0.090045,-0.015814,-0.007027,0.008321,-0.003571,-0.007779


In [602]:
#data preprocessing

df = df.drop_duplicates()

In [603]:
#splitting df data into train and test 
x_train, x_test = train_test_split(df, train_size=0.75)

In [604]:
print('Training dataframe shape:',x_train.shape)
print('Testing dataframe shape:',x_test.shape)

Training dataframe shape: (34656, 79)
Testing dataframe shape: (11553, 79)


In [605]:
# scale x_train and x_test with x_train trained scaler
scaler = StandardScaler().set_output(transform="pandas")

#removed 'label' to avoid error in scaling
x_train_scaled = scaler.fit_transform(x_train.iloc[:, :-1])
x_test_scaled = scaler.transform(x_test.iloc[:, :-1])

#adding back the "label" feature
x_train_scaled['Label'] = x_train['Label']
x_test_scaled['Label'] = x_test['Label']

print("Shape of of training dataframe after scaling:", x_train_scaled.shape)
print("Shape of of testing dataframe after scaling:", x_test_scaled.shape)



Shape of of training dataframe after scaling: (34656, 79)
Shape of of testing dataframe after scaling: (11553, 79)


In [606]:
#feature selection

#drop all columns with 0 variance
def varcheck(df: pd.DataFrame) -> pd.DataFrame:
    del_list = []
    for column in df.columns.tolist():
        if df[column].dtype == 'int64':
            if df[column].var() == 0:
                del_list.append(column)
                

    return df.drop(columns=del_list)                  

In [607]:
x_train_scaled = varcheck(x_train_scaled)
columns_kept = list(x_train_scaled.columns)
x_test_scaled = x_train_scaled[columns_kept]

#Only using varcheck() on df_train and applying the same transformations to df_test

print("Shape of of training dataframe after variance threshold:", x_train_scaled.shape)
print("Shape of of testing dataframe after variance threshold:", x_test_scaled.shape)

#no change in features...maybe scaling helped get rid of low variance columns?

Shape of of training dataframe after variance threshold: (34656, 79)
Shape of of testing dataframe after variance threshold: (34656, 79)


In [608]:
#looking for a pair of features that correlate, then drop one of them (if correlation is equal or greater to 0.95)

def corrcheck(df: pd.DataFrame) -> pd.DataFrame:
    clean_list = []
    keep_list = []
    corr_df = cleaned_df.iloc[:, :-1].corr()
    for column in corr_df.columns.tolist():
        temp_df = corr_df[column]
        for row in range(len(temp_df)):
            if temp_df[row] >= 0.95 and temp_df.index[row] != column and column not in keep_list:
                clean_list.append(column)
                keep_list.append(temp_df.index[row])                
    return df.drop(columns=clean_list)        

In [609]:
columns_kept = list()
x_train_scaled = corrcheck(x_train_scaled)
columns_kept = list(x_train_scaled.columns)
x_test_scaled = x_train_scaled[columns_kept]


print("Shape of of training dataframe after correlation correction:", x_train_scaled.shape)
print("Shape of of testing dataframe after correlation correction:", x_test_scaled.shape)

Shape of of training dataframe after correlation correction: (34656, 76)
Shape of of testing dataframe after correlation correction: (34656, 76)


/var/folders/hy/sng9twds677254718xcrwdtc0000gq/T/ipykernel_54218/871763349.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if temp_df[row] >= 0.95 and temp_df.index[row] != column and column not in keep_list:


In [611]:
#load the x_train and x_test dataframes to csv
x_train_scaled.to_csv('data/x_train_data.csv', index=False)
x_test_scaled.to_csv('data/x_test_data.csv', index=False)

In [612]:
#function to turn 'Label' to binary, attack = 1, benign = 0
def label2int(df: pd.DataFrame) -> pd.DataFrame:
    df = df.apply(lambda x: 0 if x == 'BENIGN' else 1)
    return df


In [613]:
# reading in testing and training for perceptron model

p_X_train = pd.read_csv('data/x_train_data.csv')
p_y_train = p_X_train['Label']
p_X_test = pd.read_csv('data/x_test_data.csv')
p_y_test = p_X_test['Label']


#remove label feature from training
p_X_train.drop(columns='Label', inplace=True)
p_X_test.drop(columns='Label', inplace=True)


#convert labels to binary
p_y_train = label2int(p_y_train)
p_y_test = label2int(p_y_test)

In [614]:
#perceptron model
perceptron = Perceptron(eta0=0.001, random_state=1)
perceptron = perceptron.fit(p_X_train, p_y_train)
pred = perceptron.predict(p_X_test)

In [615]:
#evaluate model prediction quality

acc_perc = accuracy_score(p_y_test, pred)
f1score_perc = f1_score(p_y_test, pred)
precision_perc = precision_score(p_y_test, pred)
recall_perc = recall_score(p_y_test, pred)

print('Perceptron Model')
print('-'*20)
print('Accuracy: {:.3f}'.format(acc_perc))
print('Precision: {:.3f}'.format(precision_perc))
print('Recall: {:.3f}'.format(recall_perc))
print('F1-score: {:.3f}'.format(f1score_perc))

Perceptron Model
--------------------
Accuracy: 0.990
Precision: 0.991
Recall: 0.999
F1-score: 0.995
